# koff evaluation

In [ ]:
import torch
import json
import random
import yaml
import numpy as np
from omegaconf import OmegaConf
from tqdm import tqdm
import os
import torchmetrics

os.chdir("/root/private_data/luog/AbAgKer")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from taming.models.AbAgKer_newLLM import AbAgKerTrainer
from main_wandb import DataModuleFromConfig, instantiate_from_config


DEVICE = torch.device("cuda:0")

def load_config(config_path, display=False):
  config = OmegaConf.load(config_path)
  if display:
    print(yaml.dump(OmegaConf.to_container(config)))
  return config

def load_model(config, ckpt_path=None):
    def init_from_ckpt(model, path, ignore_keys=list()):
      sd = torch.load(path, map_location="cpu")["state_dict"]
      keys = list(sd.keys())
      for k in keys:
          for ik in ignore_keys:
              if k.startswith(ik):
                  print("Deleting key {} from state_dict.".format(k))
                  del sd[k]
      model.load_state_dict(sd, strict=False)
      print(f"Restored from {path}") 

    model = AbAgKerTrainer(**config.model.params)
    model.to(DEVICE)
    if ckpt_path is not None:
        init_from_ckpt(model,ckpt_path)
    return model.eval()


config_path = "/root/private_data/luog/AbAgKer/infer/mutation_prediction/AA_ckpt/AbAgKer_fold2_1120_withAbAgIdata.yaml"
ckpt_path = "/root/private_data/luog/AbAgKer/infer/mutation_prediction/AA_ckpt/epoch=49-step=29350.ckpt"

config = load_config(config_path)
model = load_model(config, ckpt_path)


Using common tokenizer.
Loading pre-trained tokenizer...
Loading pre-trained tokenizer...


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at /root/private_data/luog/AbAgKer/pretrain_LLM/esm2 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028


Restored from /root/private_data/luog/AbAgKer/infer/mutation_prediction/AA_ckpt/epoch=49-step=29350.ckpt


# prediction and save result

In [ ]:

import json 
import pandas as pd
import csv
import os
from rdkit import Chem
import pandas as pd
import torch

def tensor_to_list(t):
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().numpy().tolist()
    elif isinstance(t, (list, tuple)):
        return [tensor_to_list(item) for item in t]
    elif isinstance(t, np.ndarray):
        return t.tolist()
    else:
        return t


def move_to_device(batch, device):
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, (list, tuple)):
        return type(batch)(move_to_device(x, device) for x in batch)
    elif isinstance(batch, torch.Tensor):
        return batch.to(device)
    else:
        return batch

def metrics_to_excel_row(metrics_result, prefix="val/kd_"):
    order = [
        ("ci", "CI"),
        ("spearman", "Spearman"),
        ("pearson", "Pearson"),
        ("rm2", "rm²"),
        ("rmse", "RMSE"),
        ("mse", "MSE")
    ]
    
    values = []
    for key, name in order:
        full_key = prefix + key
        if full_key in metrics_result:
            val = metrics_result[full_key]
            num = val.item() if hasattr(val, 'item') else float(val)
            values.append(f"{num:.4f}")
        else:
            values.append("N/A")
    
    header = "\t".join([name for _, name in order])
    row = "\t".join(values)
    return header, row


def data_load(pdb_name):

    data_config_path = f"/root/private_data/luog/AbAgKer/infer/mutation_prediction/data/yaml/fold2/{pdb_name}.yaml"

    data_config = load_config(data_config_path)
    data = instantiate_from_config(data_config.data)
    data.prepare_data()
    data.setup()
    val_dataloader = data.val_dataloader()
    print(f"{pdb_name} Val dataloader created with {len(val_dataloader)} batches")

    return val_dataloader

def abagkd_prediction(model, val_dataloader, save_path):
    
    model.eval()
    device = next(model.parameters()).device
    all_predictions = []

    from taming.modules.metrics.metrics import CI, RM2
    kd_metrics = torchmetrics.MetricCollection({
        "kd_pearson": torchmetrics.PearsonCorrCoef(),
        "kd_spearman": torchmetrics.SpearmanCorrCoef(),
        "kd_rm2": RM2(),
        "kd_ci": CI(),
        "kd_mse": torchmetrics.MeanSquaredError(),
        "kd_rmse": torchmetrics.MeanSquaredError(squared=False),
    }, prefix="val/")
    kd_metrics = kd_metrics.to(device)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(val_dataloader, desc="Predicting binding sites")):
            batch = move_to_device(batch,device)
            
            HLX, extra_HXL, label = model.get_input(batch)
            kd, koff, AbAgI = label[0], label[1], label[2]
            kd_pre, _, _ = model.model(HLX,extra_HXL)
            kd_metrics.update(kd_pre, kd)

            for i in range(len(label[0])):
                all_predictions.append({
                    "text": kd[i].item(),
                    "text_pre":kd_pre[i].item(),
                })
    metrics_result = kd_metrics.compute()

    header, row = metrics_to_excel_row(metrics_result)
    print(header)
    print(row)

    with open(save_path, "w", encoding='utf-8') as f:
        json.dump(all_predictions, f, indent=4, ensure_ascii=False)

    return all_predictions


if __name__ == "__main__":
    
    pdb_name = "1CZ8_19" # 1CZ8_19/1DQJ_40
    val_dataloader = data_load(pdb_name)
    # run
    predictions = abagkd_prediction(
        model=model,
        val_dataloader=val_dataloader,
        save_path=f"/root/private_data/luog/AbAgKer/infer/mutation_prediction/data/prediction_result/{pdb_name}_prediction.json"
    )

4NM8_10 Val dataloader created with 1 batches


Predicting binding sites: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]

CI	Spearman	Pearson	rm²	RMSE	MSE
0.6889	0.5152	0.6765	0.2503	1.0102	1.0206
